In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report,
                             confusion_matrix)

In [2]:
df = pd.read_csv('dataset_phishing.csv')
print(f"\n[INFO] Dataset loaded: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"[INFO] Distribusi kelas:\n{df['status'].value_counts().to_string()}")


[INFO] Dataset loaded: 11430 baris, 89 kolom
[INFO] Distribusi kelas:
status
legitimate    5715
phishing      5715


In [3]:
df_clean = df.drop(columns=['url'])
print(f"[INFO] Kolom 'url' di-drop (non-numerik)")

X = df_clean.drop(columns=['status'])
y = df_clean['status']

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"[INFO] Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"[INFO] Jumlah fitur: {X.shape[1]}")
print(f"[INFO] Missing values: {X.isnull().sum().sum()}")
scaler = StandardScaler()

[INFO] Kolom 'url' di-drop (non-numerik)
[INFO] Label encoding: {'legitimate': 0, 'phishing': 1}
[INFO] Jumlah fitur: 87
[INFO] Missing values: 0


In [4]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y_encoded,
    test_size=0.15,
    random_state=42,
    stratify=y_encoded
)
 
# Step 2: Split 70% train dan 15% validasi dari 85%
# 15/85 ≈ 0.1765 untuk mendapat split 70/15/15
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.1765,
    random_state=42,
    stratify=y_trainval
)
 
print(f"[INFO] Data Training  : {X_train.shape[0]} sampel ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"[INFO] Data Validasi  : {X_val.shape[0]} sampel ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"[INFO] Data Test      : {X_test.shape[0]} sampel ({X_test.shape[0]/len(X)*100:.1f}%)")
 
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)
print(f"[INFO] Normalisasi StandardScaler selesai (fit pada data training)")

[INFO] Data Training  : 8000 sampel (70.0%)
[INFO] Data Validasi  : 1715 sampel (15.0%)
[INFO] Data Test      : 1715 sampel (15.0%)
[INFO] Normalisasi StandardScaler selesai (fit pada data training)


In [5]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
 
cv_results = cross_validate(
    svm_model,
    X_train_scaled, y_train,
    cv=skf,
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True,
    n_jobs=-1
)
 
print(f"\n{'Fold':<6} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-" * 50)
for i in range(10):
    print(f"  {i+1:<4} "
          f"{cv_results['test_accuracy'][i]:>10.4f} "
          f"{cv_results['test_precision'][i]:>10.4f} "
          f"{cv_results['test_recall'][i]:>10.4f} "
          f"{cv_results['test_f1'][i]:>10.4f}")
 
print(f"{'Mean':<6} "
      f"{cv_results['test_accuracy'].mean():>10.4f} "
      f"{cv_results['test_precision'].mean():>10.4f} "
      f"{cv_results['test_recall'].mean():>10.4f} "
      f"{cv_results['test_f1'].mean():>10.4f}")
print(f"{'Std':<6} "
      f"{cv_results['test_accuracy'].std():>10.4f} "
      f"{cv_results['test_precision'].std():>10.4f} "
      f"{cv_results['test_recall'].std():>10.4f} "
      f"{cv_results['test_f1'].std():>10.4f}")


Fold     Accuracy  Precision     Recall   F1-Score
--------------------------------------------------
  1        0.9563     0.9620     0.9500     0.9560
  2        0.9525     0.9571     0.9475     0.9523
  3        0.9613     0.9624     0.9600     0.9612
  4        0.9563     0.9551     0.9575     0.9563
  5        0.9550     0.9619     0.9475     0.9547
  6        0.9600     0.9670     0.9525     0.9597
  7        0.9625     0.9648     0.9600     0.9624
  8        0.9537     0.9459     0.9625     0.9542
  9        0.9675     0.9770     0.9575     0.9672
  10       0.9625     0.9602     0.9650     0.9626
Mean       0.9587     0.9614     0.9560     0.9586
Std        0.0045     0.0077     0.0059     0.0045


In [6]:
svm_final = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_final.fit(X_train_scaled, y_train)
print(f"[INFO] Model SVM (RBF kernel) berhasil ditraining")
print(f"[INFO] Kernel : rbf | C : 1.0 | Gamma : scale")

[INFO] Model SVM (RBF kernel) berhasil ditraining
[INFO] Kernel : rbf | C : 1.0 | Gamma : scale


In [7]:
def evaluate(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    cm   = confusion_matrix(y_true, y_pred)
    print(f"\n--- {name} ---")
    print(f"  Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  Confusion Matrix:")
    print(f"    TP={cm[1][1]}  FP={cm[0][1]}")
    print(f"    FN={cm[1][0]}  TN={cm[0][0]}")
    return acc, prec, rec, f1

y_val_pred = svm_final.predict(X_val_scaled)
acc_v, prec_v, rec_v, f1_v = evaluate("DATA VALIDASI", y_val, y_val_pred)
 
y_test_pred = svm_final.predict(X_test_scaled)
acc_t, prec_t, rec_t, f1_t = evaluate("DATA TEST", y_test, y_test_pred)


--- DATA VALIDASI ---
  Accuracy  : 0.9539 (95.39%)
  Precision : 0.9513
  Recall    : 0.9569
  F1-Score  : 0.9541
  Confusion Matrix:
    TP=821  FP=42
    FN=37  TN=815

--- DATA TEST ---
  Accuracy  : 0.9493 (94.93%)
  Precision : 0.9498
  Recall    : 0.9487
  F1-Score  : 0.9492
  Confusion Matrix:
    TP=813  FP=43
    FN=44  TN=815


In [8]:
print(f"\n{'Metric':<12} {'CV (mean)':>12} {'Validasi':>12} {'Test':>12}")
print(f"{'Accuracy':<12} {cv_results['test_accuracy'].mean():>12.4f} {acc_v:>12.4f} {acc_t:>12.4f}")
print(f"{'Precision':<12} {cv_results['test_precision'].mean():>12.4f} {prec_v:>12.4f} {prec_t:>12.4f}")
print(f"{'Recall':<12} {cv_results['test_recall'].mean():>12.4f} {rec_v:>12.4f} {rec_t:>12.4f}")
print(f"{'F1-Score':<12} {cv_results['test_f1'].mean():>12.4f} {f1_v:>12.4f} {f1_t:>12.4f}")


Metric          CV (mean)     Validasi         Test
Accuracy           0.9587       0.9539       0.9493
Precision          0.9614       0.9513       0.9498
Recall             0.9560       0.9569       0.9487
F1-Score           0.9586       0.9541       0.9492


In [9]:
import joblib

joblib.dump(svm_final, 'svm_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(list(X.columns), 'feature_names.pkl')

['feature_names.pkl']